# Notebook 07 · Generate and preserve model outputs for fresh tasks

Fresh-run release. Follow the input and review gates in RUN_ORDER.md.

In [1]:
# ============================================================
# NOTEBOOK 07 — CELL 01
# Discover frozen Stage 00 and Stage 06 input datasets.
# ============================================================

from __future__ import annotations

from collections import Counter, defaultdict
from pathlib import Path
import csv
import hashlib
import io
import json
import os
import tempfile
import time


CELL01_STARTED = time.time()
STAGE07_VERSION = "stage07-fresh-pilot-v1"
SOURCE_BASE = Path(os.environ.get("STAGE07_SOURCE_BASE", "/kaggle/input"))
PROJECT_ROOT = Path(os.environ.get(
    "STAGE07_PROJECT_ROOT", "/kaggle/working/prose_operation_experiment_stage07_fresh"
))
DATASET_ROOT = Path(os.environ.get(
    "STAGE07_DATASET_ROOT", "/kaggle/working/stage_07_fresh_dataset"
))
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_ROOT.mkdir(parents=True, exist_ok=True)


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_json_bytes(value) -> bytes:
    return (json.dumps(value, ensure_ascii=False, sort_keys=True,
                       separators=(",", ":")) + "\n").encode("utf-8")


def write_bytes_immutable(path: Path, data: bytes) -> str:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if path.read_bytes() != data:
            raise RuntimeError(f"Immutable artifact differs: {path}")
        return "REUSED_IDENTICAL"
    temporary = path.with_name(path.name + ".tmp-" + str(os.getpid()))
    try:
        with temporary.open("xb") as stream:
            stream.write(data)
            stream.flush()
            os.fsync(stream.fileno())
        if path.exists():
            if path.read_bytes() != data:
                raise RuntimeError(f"Immutable artifact changed: {path}")
            return "REUSED_IDENTICAL"
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()
    return "WRITTEN"


def write_json_immutable(path: Path, value) -> str:
    return write_bytes_immutable(path, canonical_json_bytes(value))


def write_json_atomic(path: Path, value) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp-" + str(os.getpid()))
    try:
        with temporary.open("wb") as stream:
            stream.write(canonical_json_bytes(value))
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


def read_json(path: Path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines()
            if line.strip()]


def safe_relative(value: str) -> Path:
    candidate = Path(value)
    if (candidate.is_absolute() or ".." in candidate.parts
            or not candidate.parts or candidate.parts[0] == "."):
        raise RuntimeError(f"Unsafe relative artifact path: {value!r}")
    return candidate


def report_cell(number: int, **details):
    report = {"status": "PASS", "notebook": "07", "cell": number,
              "version": STAGE07_VERSION, **details}
    destination = PROJECT_ROOT / "reports/audit" / f"cell{number:02d}_stage07.json"
    # Runtime availability and checkpoint progress change across Kaggle sessions.
    if number in (3, 8, 9):
        write_json_atomic(destination, report)
    else:
        write_json_immutable(destination, report)
    print(json.dumps(report, indent=2))
    print(f"\nNOTEBOOK 07 — CELL {number:02d}: PASS")
    return report


def require_cell_pass(number: int):
    report_path = PROJECT_ROOT / "reports/audit" / f"cell{number:02d}_stage07.json"
    if not report_path.is_file() or read_json(report_path).get("status") != "PASS":
        raise RuntimeError(f"Required Cell {number:02d} has not passed.")


def find_frozen_stage(stage: str):
    expected_external = f"stage_{stage}_manifest.json"
    if not SOURCE_BASE.is_dir():
        raise RuntimeError(f"Kaggle input directory not found: {SOURCE_BASE}")
    matches = []
    for registry in SOURCE_BASE.rglob(f"registries/{stage}_manifest.json"):
        root = registry.parent.parent
        internal = read_json(registry)
        if (internal.get("stage") != stage
                or internal.get("status", internal.get("stage_status")) != "PASS"):
            continue
        for ancestor in root.parents:
            external_path = ancestor / expected_external
            if external_path.is_file():
                external = read_json(external_path)
                if external.get("stage") == stage and external.get("status") == "PASS":
                    matches.append((root, registry, external_path, internal, external))
                break
    if len(matches) != 1:
        raise RuntimeError(
            f"Stage {stage}: expected exactly one manifest-identified extracted root, "
            f"found {len(matches)}. Input directory: {SOURCE_BASE}"
        )
    root, registry, external_path, internal, external = matches[0]
    if sha256_file(registry) != external["internal_manifest_sha256"]:
        raise RuntimeError(f"Stage {stage}: internal manifest hash mismatch")
    return {"root": root, "registry_path": registry, "external_path": external_path,
            "internal": internal, "external": external}


STAGES = {stage: find_frozen_stage(stage) for stage in ("00", "06")}
if STAGES["06"]["internal"].get("design_id") != "openapi-fresh-pilot-v1":
    raise RuntimeError("Attach the fresh Stage 06 release.")
if STAGES["06"]["internal"]["upstream_manifest_sha256"]["00"] != sha256_file(STAGES["00"]["registry_path"]):
    raise RuntimeError("The attached Stage 00 is not the input used to build this Stage 06.")
if STAGES["06"]["internal"].get("operation_count") != 100:
    raise RuntimeError("Stage 06 does not declare exactly 100 operations")
if STAGES["06"]["internal"].get("task_count") != 200:
    raise RuntimeError("Stage 06 does not declare exactly 200 tasks")
if STAGES["06"]["internal"].get("evaluation_worker_version") != "stage06-isolated-worker-v11-single-opportunity":
    raise RuntimeError("Attach only the corrected Stage 06 v4 dataset")
if not STAGES["06"]["internal"].get("checkpoint_b_inherited"):
    raise RuntimeError("Stage 06 checkpoint B was not inherited")
if STAGES["06"]["internal"].get("model_generation_performed"):
    raise RuntimeError("Stage 06 must not already contain model generation")
if not STAGES["06"]["external"].get("may_begin_notebook_07"):
    raise RuntimeError("Stage 06 has not authorized Notebook 07")

SOURCE_IDENTITIES = {
    s: {"root": str(v["root"]), "internal_manifest_sha256": sha256_file(v["registry_path"]),
        "external_manifest_sha256": sha256_file(v["external_path"])}
    for s, v in STAGES.items()
}
report_cell(1, stage00_root=str(STAGES["00"]["root"]),
            stage06_root=str(STAGES["06"]["root"]), operation_count=100,
            expected_task_count=200, checkpoint_b_reached=True)


{
  "status": "PASS",
  "notebook": "07",
  "cell": 1,
  "version": "stage07-fresh-pilot-v1",
  "stage00_root": "/kaggle/input/datasets/anonuser01/stage-00-provenance/stage_00_provenance_v1/prose_operation_experiment",
  "stage06_root": "/kaggle/input/datasets/anonuser01/stage-06/stage_06_fresh/stage_06_fresh/prose_operation_experiment",
  "operation_count": 100,
  "expected_task_count": 200,
  "checkpoint_b_reached": true
}

NOTEBOOK 07 — CELL 01: PASS


{'status': 'PASS',
 'notebook': '07',
 'cell': 1,
 'version': 'stage07-fresh-pilot-v1',
 'stage00_root': '/kaggle/input/datasets/anonuser01/stage-00-provenance/stage_00_provenance_v1/prose_operation_experiment',
 'stage06_root': '/kaggle/input/datasets/anonuser01/stage-06/stage_06_fresh/stage_06_fresh/prose_operation_experiment',
 'operation_count': 100,
 'expected_task_count': 200,
 'checkpoint_b_reached': True}

In [2]:
# ============================================================
# NOTEBOOK 07 — CELL 02
# Verify all upstream artifact hashes and the 200-task contract.
# ============================================================

from __future__ import annotations

from collections import defaultdict
import csv
import io
import json


import hashlib
import json
import re
from pathlib import PurePosixPath


ARTIFACT_SCHEMA = "openapi-artifact-v1"
PRODUCER_NOTEBOOKS = {
    "00": "00_protocol_and_runtime",
    "01": "01_corpus_preparation_and_eligibility",
    "02": "02_operation_slicing_and_prose_measurement",
    "03": "03_pair_and_evaluator_feasibility",
    "04": "04_strata_and_pilot_selection",
    "05": "05_oracle_build_and_review",
    "06": "06_build_corrected_evaluator_and_generation_tasks",
    "07": "07_pilot_generation",
    "08": "08_pilot_execution_and_review",
}


def provenance_digest(value):
    data = json.dumps(value, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=False, allow_nan=False).encode("utf-8")
    return hashlib.sha256(data).hexdigest()


def artifact_path_for(record):
    paths = {record[key] for key in ("artifact_path", "relative_path", "path")
             if key in record}
    if len(paths) != 1:
        raise RuntimeError("Missing or inconsistent artifact paths")
    name = paths.pop()
    path = PurePosixPath(name)
    if not name or name == "." or path.is_absolute() or ".." in path.parts or "\\" in name or path.as_posix() != name:
        raise RuntimeError(f"Invalid artifact path: {name!r}")
    return name


def manifest_digest_valid(value):
    return isinstance(value, str) and re.fullmatch(r"[0-9a-f]{64}", value) is not None


def register_artifact_provenance(manifest):
    stage = manifest["stage"]
    artifacts = manifest["artifacts"]
    if "upstream_manifest_sha256" not in manifest:
        if stage != "00":
            raise RuntimeError(f"Stage {stage} has no upstream identity")
        manifest["upstream_manifest_sha256"] = {}
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest.get("configuration_hash")
    if config_hash is None:
        config_hash = provenance_digest({artifact_path_for(row): row["sha256"]
                                         for row in artifacts
                                         if artifact_path_for(row).startswith("config/")})
    manifest["configuration_hash"] = config_hash
    manifest["artifact_record_schema"] = ARTIFACT_SCHEMA
    upstream_hash = provenance_digest(links)
    for row in artifacts:
        row.update(artifact_path=artifact_path_for(row), schema_version=ARTIFACT_SCHEMA,
                   producer_notebook=PRODUCER_NOTEBOOKS[stage], config_hash=config_hash,
                   upstream_manifest_hash=upstream_hash)
    verify_artifact_provenance(manifest)


def verify_artifact_provenance(manifest):
    stage = manifest["stage"]
    if manifest.get("artifact_record_schema") != ARTIFACT_SCHEMA:
        raise RuntimeError(f"Stage {stage} requires a corrected provenance export")
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest["configuration_hash"]
    if (not isinstance(links, dict) or bool(links) != (stage != "00")
            or any(key not in PRODUCER_NOTEBOOKS or key >= stage or not manifest_digest_valid(value)
                   for key, value in links.items()) or not manifest_digest_valid(config_hash)):
        raise RuntimeError(f"Invalid Stage {stage} provenance")
    expected = (ARTIFACT_SCHEMA, PRODUCER_NOTEBOOKS[stage], config_hash, provenance_digest(links))
    seen = set()
    artifacts = manifest["artifacts"]
    if not artifacts or len(artifacts) != manifest["artifact_count"]:
        raise RuntimeError(f"Invalid Stage {stage} artifact count")
    for row in artifacts:
        name = artifact_path_for(row)
        actual = tuple(row.get(key) for key in
                       ("schema_version", "producer_notebook", "config_hash", "upstream_manifest_hash"))
        if (row.get("artifact_path") != name or name in seen or actual != expected
                or not manifest_digest_valid(row.get("sha256"))
                or type(row.get("bytes")) is not int or row["bytes"] < 0):
            raise RuntimeError(f"Invalid Stage {stage} artifact provenance: {name}")
        seen.add(name)


CELL02_STARTED = time.time()
require_cell_pass(1)
verified_counts = {}
for stage, source in STAGES.items():
    verify_artifact_provenance(source["internal"])
    records = source["internal"].get("artifacts", [])
    if len(records) != source["internal"].get("artifact_count"):
        raise RuntimeError(f"Stage {stage}: registered artifact count mismatch")
    seen = set()
    for item in records:
        relative = item.get("relative_path", item.get("artifact_path"))
        safe_relative(relative)
        if relative in seen:
            raise RuntimeError(f"Stage {stage}: duplicate manifest path: {relative}")
        seen.add(relative)
        path = source["root"] / relative
        if not path.is_file() or path.is_symlink():
            raise RuntimeError(f"Stage {stage}: missing or symlinked file {relative}")
        if path.stat().st_size != item["bytes"] or sha256_file(path) != item["sha256"]:
            raise RuntimeError(f"Stage {stage}: content hash/byte mismatch {relative}")
    verified_counts[stage] = len(records)

task_path = STAGES["06"]["root"] / "data/tasks.csv"
TASK_LEDGER_BYTES = task_path.read_bytes()
TASKS = list(csv.DictReader(io.StringIO(TASK_LEDGER_BYTES.decode("utf-8"))))
EXPECTED_COLUMNS = (
    "task_id", "pair_id", "op_id", "partition", "condition", "replicate",
    "seed", "order", "model_revision", "prompt_path", "prompt_sha256",
    "prompt_tokens", "expected_output_path",
)
if tuple(TASKS[0]) != EXPECTED_COLUMNS or len(TASKS) != 200:
    raise RuntimeError("Stage 06 task ledger must contain 200 rows and 13 exact columns")
if len({r["task_id"] for r in TASKS}) != 200:
    raise RuntimeError("Duplicate task identifiers")
if len({r["expected_output_path"] for r in TASKS}) != 200:
    raise RuntimeError("Duplicate expected output paths")
if sorted(int(r["order"]) for r in TASKS) != list(range(1, 201)):
    raise RuntimeError("Expected task orders 1..200")
TASKS.sort(key=lambda row: int(row["order"]))
pair_groups = defaultdict(list)
for task in TASKS:
    if task["partition"] != "pilot" or task["condition"] not in ("L0", "L1"):
        raise RuntimeError("Invalid partition or condition in task ledger")
    if task["replicate"] != "1" or int(task["seed"]) < 0:
        raise RuntimeError("Unexpected replicate or seed")
    if (task["expected_output_path"] !=
            f"runs/pilot/raw/{task['task_id']}.txt"):
        raise RuntimeError("Task has an unexpected output destination")
    safe_relative(task["expected_output_path"])
    safe_relative(task["prompt_path"])
    if task["model_revision"] != STAGES["00"]["external"]["model_revision"]:
        raise RuntimeError("Task revision differs from frozen Stage 00 model")
    prompt_path = STAGES["06"]["root"] / task["prompt_path"]
    if sha256_file(prompt_path) != task["prompt_sha256"]:
        raise RuntimeError("Prompt SHA-256 mismatch: " + task["task_id"])
    if not 0 < int(task["prompt_tokens"]) <= 5120:
        raise RuntimeError("Invalid prompt-token count")
    pair_groups[task["pair_id"]].append(task)
if len(pair_groups) != 100 or len({r["op_id"] for r in TASKS}) != 100:
    raise RuntimeError("Expected 100 unique paired operations")
for pair, group in pair_groups.items():
    if (len(group) != 2 or {r["condition"] for r in group} != {"L0", "L1"}
            or len({r["seed"] for r in group}) != 1
            or len({r["op_id"] for r in group}) != 1):
        raise RuntimeError("Unbalanced or unpaired condition: " + pair)

UPSTREAM_VERIFICATION = {
    "status": "PASS", "version": STAGE07_VERSION,
    "verified_artifacts": verified_counts, "verified_total": sum(verified_counts.values()),
    "task_count": 200, "pair_count": 100,
    "task_ledger_sha256": sha256_bytes(TASK_LEDGER_BYTES),
    "sources": SOURCE_IDENTITIES,
}
write_json_immutable(PROJECT_ROOT / "reports/audit/upstream_verification.json",
                     UPSTREAM_VERIFICATION)
report_cell(2, verified_artifact_count=sum(verified_counts.values()),
            operation_pairs=100, task_count=200,
            task_ledger_sha256=sha256_bytes(TASK_LEDGER_BYTES))


{
  "status": "PASS",
  "notebook": "07",
  "cell": 2,
  "version": "stage07-fresh-pilot-v1",
  "verified_artifact_count": 380,
  "operation_pairs": 100,
  "task_count": 200,
  "task_ledger_sha256": "b3170527b07cd39dbaf4cdc185d0f1e9e58c48f675dba33250adc9375467b235"
}

NOTEBOOK 07 — CELL 02: PASS


{'status': 'PASS',
 'notebook': '07',
 'cell': 2,
 'version': 'stage07-fresh-pilot-v1',
 'verified_artifact_count': 380,
 'operation_pairs': 100,
 'task_count': 200,
 'task_ledger_sha256': 'b3170527b07cd39dbaf4cdc185d0f1e9e58c48f675dba33250adc9375467b235'}

In [3]:
# ============================================================
# NOTEBOOK 07 — CELL 03
# Qualify the dual-T4 CUDA runtime and immutable model access.
# ============================================================

from __future__ import annotations

from importlib import metadata
import importlib
import json
import os
import platform
import shutil
import time

import yaml


CELL03_STARTED = time.time()
require_cell_pass(2)
RUNTIME = yaml.safe_load((STAGES["00"]["root"] / "config/runtime.yaml").read_text())
MODEL_ID = RUNTIME["model_id"]
MODEL_REVISION = RUNTIME["resolved_model_revision"]
TOKENIZER_REVISION = RUNTIME["resolved_tokenizer_revision"]
if MODEL_REVISION != TOKENIZER_REVISION:
    raise RuntimeError("Frozen model/tokenizer revisions differ")
if MODEL_ID != "Qwen/Qwen2.5-Coder-7B-Instruct":
    raise RuntimeError("Unexpected frozen model ID")
if RUNTIME["dtype"] != "float16" or RUNTIME["max_total_tokens"] != 17408:
    raise RuntimeError("Unexpected frozen precision/context configuration")

PACKAGE_VERSIONS = {}
for package in ("torch", "transformers", "accelerate", "safetensors",
                "tokenizers", "huggingface-hub", "PyYAML"):
    try:
        PACKAGE_VERSIONS[package] = metadata.version(package)
    except metadata.PackageNotFoundError as error:
        raise RuntimeError(f"Missing required Kaggle package: {package}") from error

# All GPU-sensitive libraries are imported after the frozen configuration.
import torch
from huggingface_hub import HfApi, snapshot_download

if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError("HARDWARE_BLOCK: Notebook 07 requires Kaggle GPU T4 x2")
GPU_RECORDS = []
for index in (0, 1):
    properties = torch.cuda.get_device_properties(index)
    free_bytes, total_bytes = torch.cuda.mem_get_info(index)
    if "T4" not in properties.name.upper():
        raise RuntimeError("HARDWARE_BLOCK: expected Tesla T4, found " + properties.name)
    if free_bytes < 9 * (1024 ** 3):
        raise RuntimeError("HARDWARE_BLOCK: insufficient free VRAM on GPU " + str(index))
    GPU_RECORDS.append({"index": index, "name": properties.name,
                        "total_gib": round(total_bytes / (1024 ** 3), 3),
                        "free_gib": round(free_bytes / (1024 ** 3), 3)})

free_disk_gib = shutil.disk_usage(PROJECT_ROOT).free / (1024 ** 3)
cache_available = False
try:
    cached_path = Path(snapshot_download(
        repo_id=MODEL_ID, revision=MODEL_REVISION, local_files_only=True,
    ))
    cached_weight_bytes = sum(
        p.stat().st_size for p in cached_path.glob("*.safetensors") if p.is_file()
    )
    cache_available = cached_weight_bytes >= 10 * (1024 ** 3)
except Exception:
    pass
access_mode = "FROZEN_REVISION_CACHED" if cache_available else "FROZEN_REVISION_REMOTE"
if not cache_available:
    try:
        model_info = HfApi().model_info(repo_id=MODEL_ID, revision=MODEL_REVISION)
    except Exception as error:
        raise RuntimeError(
            "MODEL_ACCESS_BLOCK: The frozen Hugging Face model is not cached "
            "and cannot be accessed. Enable Kaggle Internet or attach a trusted snapshot."
        ) from error
    if model_info.sha != MODEL_REVISION:
        raise RuntimeError("Model registry returned an unexpected revision")
    if free_disk_gib < 17.0:
        raise RuntimeError("STORAGE_BLOCK: at least 17 GiB of free cache disk required")

ENVIRONMENT_RECORD = {
    "status": "PASS", "stage": "07", "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION, "tokenizer_revision": TOKENIZER_REVISION,
    "python": platform.python_version(), "torch_cuda": torch.version.cuda,
    "package_versions": PACKAGE_VERSIONS, "gpu_devices": GPU_RECORDS,
    "free_disk_gib": round(free_disk_gib, 3), "model_access": access_mode,
}
write_json_atomic(PROJECT_ROOT / "reports/runtime/environment_preflight.json",
                  ENVIRONMENT_RECORD)
report_cell(3, gpu_count=2, gpu_names=[g["name"] for g in GPU_RECORDS],
            model_revision=MODEL_REVISION, model_access=access_mode,
            free_disk_gib=round(free_disk_gib, 2))


{
  "status": "PASS",
  "notebook": "07",
  "cell": 3,
  "version": "stage07-fresh-pilot-v1",
  "gpu_count": 2,
  "gpu_names": [
    "Tesla T4",
    "Tesla T4"
  ],
  "model_revision": "c03e6d358207e414f1eca0bb1891e29f1db0e242",
  "model_access": "FROZEN_REVISION_REMOTE",
  "free_disk_gib": 19.5
}

NOTEBOOK 07 — CELL 03: PASS


{'status': 'PASS',
 'notebook': '07',
 'cell': 3,
 'version': 'stage07-fresh-pilot-v1',
 'gpu_count': 2,
 'gpu_names': ['Tesla T4', 'Tesla T4'],
 'model_revision': 'c03e6d358207e414f1eca0bb1891e29f1db0e242',
 'model_access': 'FROZEN_REVISION_REMOTE',
 'free_disk_gib': 19.5}

In [4]:
# ============================================================
# NOTEBOOK 07 — CELL 04
# Restore only frozen configuration, task inputs and prompts.
# ============================================================

from __future__ import annotations

import json
import time


CELL04_STARTED = time.time()
require_cell_pass(3)
restore_list = [
    ("00", "config/runtime.yaml", "config/runtime.yaml"),
    ("00", "config/system_prompt.txt", "config/system_prompt.txt"),
    ("00", "config/user_prompt.txt", "config/user_prompt.txt"),
    ("00", "config/study.yaml", "config/study.yaml"),
    ("00", "reports/runtime/extraction_policy.json", "config/extraction_policy.json"),
    ("00", "reports/runtime/extraction_qualification.json",
     "reports/qualification/stage00_extraction.json"),
    ("00", "reports/runtime/development_prompt.txt",
     "data/development/development_prompt.txt"),
    ("06", "config/analysis_registered.yaml", "config/analysis_registered.yaml"),
    ("06", "config/metrics.yaml", "config/metrics.yaml"),
    ("06", "data/tasks.csv", "data/tasks.csv"),
    ("06", "data/audits/cell15_prompt_token_ledger.jsonl",
     "data/audits/cell15_prompt_token_ledger.jsonl"),
    ("06", "registries/06_frozen_dependency_hashes.json",
     "registries/06_frozen_dependency_hashes.json"),
]
for task in TASKS:
    restore_list.append(("06", task["prompt_path"], task["prompt_path"]))

restoration_records = []
for stage, source_relative, destination_relative in restore_list:
    src = STAGES[stage]["root"] / safe_relative(source_relative)
    dest = PROJECT_ROOT / safe_relative(destination_relative)
    if not src.is_file():
        raise RuntimeError("Missing qualified input file: " + str(src))
    data = src.read_bytes()
    write_bytes_immutable(dest, data)
    restoration_records.append({"stage": stage, "source": source_relative,
                                "destination": destination_relative,
                                "sha256": sha256_bytes(data)})

if len(restoration_records) != 212:
    raise RuntimeError("Expected 12 frozen configuration files and 200 prompts")
write_json_immutable(PROJECT_ROOT / "reports/audit/restored_inputs.json",
                     {"status": "PASS", "records": restoration_records})
report_cell(4, restored_file_count=len(restoration_records), restored_prompts=200,
            upstream_sources=["00", "06"], model_generation_performed=False)


{
  "status": "PASS",
  "notebook": "07",
  "cell": 4,
  "version": "stage07-fresh-pilot-v1",
  "restored_file_count": 212,
  "restored_prompts": 200,
  "upstream_sources": [
    "00",
    "06"
  ],
  "model_generation_performed": false
}

NOTEBOOK 07 — CELL 04: PASS


{'status': 'PASS',
 'notebook': '07',
 'cell': 4,
 'version': 'stage07-fresh-pilot-v1',
 'restored_file_count': 212,
 'restored_prompts': 200,
 'upstream_sources': ['00', '06'],
 'model_generation_performed': False}

In [5]:
# ============================================================
# NOTEBOOK 07 — CELL 05
# Load exact pinned Qwen model and verify all Stage 06 prompt IDs.
# ============================================================

from __future__ import annotations

import json
import time

from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer


CELL05_STARTED = time.time()
require_cell_pass(4)
load_started = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=TOKENIZER_REVISION, trust_remote_code=False,
)
model_config = AutoConfig.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False,
)
context_candidates = [
    getattr(model_config, name, None)
    for name in ("max_position_embeddings", "n_positions", "max_sequence_length", "seq_length")
]
if not any(isinstance(x, int) and x >= 17408 for x in context_candidates):
    raise RuntimeError("MODEL_BLOCK: frozen model declares insufficient context")
if not tokenizer.chat_template:
    raise RuntimeError("MODEL_BLOCK: missing frozen chat template")
if tokenizer.pad_token_id is None:
    if tokenizer.eos_token_id is None:
        raise RuntimeError("MODEL_BLOCK: model lacks EOS and PAD token IDs")
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False,
    dtype=torch.float16, device_map="auto",
    max_memory={0: "13GiB", 1: "13GiB", "cpu": "20GiB"},
    low_cpu_mem_usage=True,
)
model.eval()
device_map = {str(name): str(device) for name, device
              in getattr(model, "hf_device_map", {}).items()}
if not device_map:
    raise RuntimeError("MODEL_BLOCK: model was not loaded with an explicit device map")
if any(value in ("cpu", "disk") for value in device_map.values()):
    raise RuntimeError("MODEL_BLOCK: FP16 weights were offloaded outside the GPUs")
assigned_gpus = set()
for device in device_map.values():
    if device in ("0", "cuda:0"):
        assigned_gpus.add(0)
    if device in ("1", "cuda:1"):
        assigned_gpus.add(1)
if assigned_gpus != {0, 1}:
    raise RuntimeError("MODEL_BLOCK: model was not distributed across both Tesla T4s")
if next(model.parameters()).dtype != torch.float16:
    raise RuntimeError("MODEL_BLOCK: parameter precision is not FP16")
embedding_device = model.get_input_embeddings().weight.device
if embedding_device.type != "cuda":
    raise RuntimeError("MODEL_BLOCK: input embedding device is not CUDA")

TOKEN_LEDGER = read_jsonl(PROJECT_ROOT / "data/audits/cell15_prompt_token_ledger.jsonl")
ledger_lookup = {(r["op_id"], r["condition"]): r for r in TOKEN_LEDGER}
if len(TOKEN_LEDGER) != 200 or len(ledger_lookup) != 200:
    raise RuntimeError("Stage 06 token ledger must contain 200 distinct entries")
for task in TASKS:
    prompt = (PROJECT_ROOT / task["prompt_path"]).read_text(encoding="utf-8")
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    expected = ledger_lookup[(task["op_id"], task["condition"])]
    actual_token_hash = sha256_bytes(json.dumps(
        ids, separators=(",", ":")
    ).encode("utf-8"))
    if (actual_token_hash != expected["token_ids_sha256"]
            or len(ids) != int(task["prompt_tokens"])):
        raise RuntimeError("MODEL_BLOCK: Stage 06 token IDs changed for " + task["task_id"])

MODEL_LOAD_RECORD = {
    "status": "PASS", "model_id": MODEL_ID, "revision": MODEL_REVISION,
    "tokenizer_revision": TOKENIZER_REVISION, "dtype": "float16",
    "device_map": device_map, "input_device": str(embedding_device),
    "parameter_count": sum(p.numel() for p in model.parameters()),
    "loading_seconds": round(time.perf_counter() - load_started, 3),
    "prompts_retokenized_and_verified": 200,
    "versions": PACKAGE_VERSIONS,
}
write_json_atomic(PROJECT_ROOT / "reports/runtime/model_load_qualification.json",
                  MODEL_LOAD_RECORD)
report_cell(5, model_revision=MODEL_REVISION, prompt_tokens_verified=200,
            precision="FP16", cuda_device_map=device_map)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

{
  "status": "PASS",
  "notebook": "07",
  "cell": 5,
  "version": "stage07-fresh-pilot-v1",
  "model_revision": "c03e6d358207e414f1eca0bb1891e29f1db0e242",
  "prompt_tokens_verified": 200,
  "precision": "FP16",
  "cuda_device_map": {
    "model.embed_tokens": "0",
    "model.layers.0": "0",
    "model.layers.1": "0",
    "model.layers.2": "0",
    "model.layers.3": "0",
    "model.layers.4": "0",
    "model.layers.5": "0",
    "model.layers.6": "0",
    "model.layers.7": "0",
    "model.layers.8": "0",
    "model.layers.9": "0",
    "model.layers.10": "0",
    "model.layers.11": "0",
    "model.layers.12": "0",
    "model.layers.13": "0",
    "model.layers.14": "1",
    "model.layers.15": "1",
    "model.layers.16": "1",
    "model.layers.17": "1",
    "model.layers.18": "1",
    "model.layers.19": "1",
    "model.layers.20": "1",
    "model.layers.21": "1",
    "model.layers.22": "1",
    "model.layers.23": "1",
    "model.layers.24": "1",
    "model.layers.25": "1",
    "model.lay

{'status': 'PASS',
 'notebook': '07',
 'cell': 5,
 'version': 'stage07-fresh-pilot-v1',
 'model_revision': 'c03e6d358207e414f1eca0bb1891e29f1db0e242',
 'prompt_tokens_verified': 200,
 'precision': 'FP16',
 'cuda_device_map': {'model.embed_tokens': '0',
  'model.layers.0': '0',
  'model.layers.1': '0',
  'model.layers.2': '0',
  'model.layers.3': '0',
  'model.layers.4': '0',
  'model.layers.5': '0',
  'model.layers.6': '0',
  'model.layers.7': '0',
  'model.layers.8': '0',
  'model.layers.9': '0',
  'model.layers.10': '0',
  'model.layers.11': '0',
  'model.layers.12': '0',
  'model.layers.13': '0',
  'model.layers.14': '1',
  'model.layers.15': '1',
  'model.layers.16': '1',
  'model.layers.17': '1',
  'model.layers.18': '1',
  'model.layers.19': '1',
  'model.layers.20': '1',
  'model.layers.21': '1',
  'model.layers.22': '1',
  'model.layers.23': '1',
  'model.layers.24': '1',
  'model.layers.25': '1',
  'model.layers.26': '1',
  'model.layers.27': '1',
  'model.norm': '1',
  'model

In [6]:
# ============================================================
# NOTEBOOK 07 — CELL 06
# Repeated seeded development-only generation, with exact token-ID match.
# ============================================================

from __future__ import annotations

import random
import time
import json


CELL06_STARTED = time.time()
require_cell_pass(5)
DEVELOPMENT_PROMPT = (
    PROJECT_ROOT / "data/development/development_prompt.txt"
).read_text(encoding="utf-8")
development_ids = tokenizer.encode(DEVELOPMENT_PROMPT, add_special_tokens=False)
if len(development_ids) > 5120 or not development_ids:
    raise RuntimeError("Unexpected Stage 00 development-prompt token budget")
GENERATION_SETTINGS = dict(RUNTIME["generation"])
GENERATION_SETTINGS.pop("generation_seed", None)
GENERATION_SETTINGS.pop("replicates_per_condition", None)
GENERATION_SETTINGS.pop("dtype", None)
seed = int(RUNTIME["generation"]["generation_seed"])


def development_generation():
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    inputs = torch.tensor([development_ids], dtype=torch.long, device=embedding_device)
    mask = torch.ones_like(inputs)
    with torch.inference_mode():
        sequence = model.generate(
            input_ids=inputs, attention_mask=mask,
            max_new_tokens=64, do_sample=GENERATION_SETTINGS["do_sample"],
            temperature=GENERATION_SETTINGS["temperature"],
            top_p=GENERATION_SETTINGS["top_p"],
            top_k=GENERATION_SETTINGS["top_k"],
            repetition_penalty=GENERATION_SETTINGS["repetition_penalty"],
            num_beams=GENERATION_SETTINGS["num_beams"],
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True, return_dict_in_generate=False, output_scores=False,
        )
    torch.cuda.synchronize()
    return sequence[0, len(development_ids):].detach().cpu().tolist()


first_ids = development_generation()
second_ids = development_generation()
if not first_ids or first_ids != second_ids:
    raise RuntimeError(
        "REPRODUCIBILITY_BLOCK: duplicate development generations returned "
        "different token IDs. Do not begin experimental generation."
    )
SEED_QUALIFICATION = {
    "status": "PASS", "seed": seed, "input_tokens": len(development_ids),
    "qualification_max_new_tokens": 64,
    "generated_tokens": len(first_ids),
    "generated_token_ids_sha256": sha256_bytes(json.dumps(
        first_ids, separators=(",", ":")
    ).encode("utf-8")),
    "two_generations_identical": True,
    "experimental_tasks_executed": 0,
}
write_json_immutable(PROJECT_ROOT / "reports/qualification/seeded_generation.json",
                     SEED_QUALIFICATION)
report_cell(6, identical_development_generations=True,
            development_generated_tokens=len(first_ids),
            experimental_tasks_executed=0)


{
  "status": "PASS",
  "notebook": "07",
  "cell": 6,
  "version": "stage07-fresh-pilot-v1",
  "identical_development_generations": true,
  "development_generated_tokens": 64,
  "experimental_tasks_executed": 0
}

NOTEBOOK 07 — CELL 06: PASS


{'status': 'PASS',
 'notebook': '07',
 'cell': 6,
 'version': 'stage07-fresh-pilot-v1',
 'identical_development_generations': True,
 'development_generated_tokens': 64,
 'experimental_tasks_executed': 0}

In [7]:
# ============================================================
# NOTEBOOK 07 — CELL 07
# Implement and qualify the visible, versioned generation module.
# ============================================================

# BEGIN_STAGE07_GENERATION_MODULE

from __future__ import annotations

import hashlib
import json
import os
import random
import re
import time
from pathlib import Path


GENERATION_MODULE_VERSION = "stage07-generation-v1-frozen-paired"
OUTER_PYTHON_FENCE = re.compile(
    r"\A\s*```python[ \t]*\r?\n(?P<code>[\s\S]*?)\r?\n```\s*\Z",
    flags=re.IGNORECASE,
)


def module_hash(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def module_json_bytes(value) -> bytes:
    return (json.dumps(value, sort_keys=True, ensure_ascii=False,
                       separators=(",", ":")) + "\n").encode("utf-8")


def atomic_replace_bytes(path, data: bytes):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp-" + str(os.getpid()))
    try:
        with temporary.open("wb") as stream:
            stream.write(data)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


def immutable_bytes(path, data: bytes):
    path = Path(path)
    if path.exists():
        if path.read_bytes() != data:
            raise RuntimeError("Immutable raw artifact differs: " + str(path))
        return "REUSED_IDENTICAL"
    atomic_replace_bytes(path, data)
    return "WRITTEN"


def write_task_checkpoint(path, record):
    atomic_replace_bytes(path, module_json_bytes(record))


def validate_task_checkpoint(record, task, fingerprint):
    if record["task_id"] != task["task_id"]:
        raise RuntimeError("Checkpoint task identity mismatch")
    if record["fingerprint"] != fingerprint:
        raise RuntimeError("STALE_CHECKPOINT_BLOCK: generation runtime/config changed")
    if record["prompt_sha256"] != task["prompt_sha256"]:
        raise RuntimeError("STALE_CHECKPOINT_BLOCK: prompt hash changed")
    if record["expected_output_path"] != task["expected_output_path"]:
        raise RuntimeError("STALE_CHECKPOINT_BLOCK: output location changed")
    if record["task_sha256"] != module_hash(module_json_bytes(task)):
        raise RuntimeError("STALE_CHECKPOINT_BLOCK: task row changed")
    if record["state"] not in ("STARTED", "GENERATED", "RAW_RECOVERED",
                               "NO_OUTPUT_RUNTIME_FAILURE"):
        raise RuntimeError("Unknown checkpoint state")
    if not isinstance(record["attempts"], list) or len(record["attempts"]) > 2:
        raise RuntimeError("Invalid checkpoint attempt history")


def extract_python_module(raw_text: str):
    """The exact Stage 00 extraction policy, with no repairs or compilation."""
    match = OUTER_PYTHON_FENCE.fullmatch(raw_text)
    if match:
        code = match.group("code")
        if "```" in code:
            return None, "rejected", "nested_or_multiple_fences"
        return code, "accepted_outer_python_fence", None
    if "```" in raw_text:
        return None, "rejected", "mixed_or_multiple_fences"
    if not raw_text.strip():
        return None, "rejected", "empty_output"
    return raw_text, "accepted_raw_python", None


def generated_continuation(model, tokenizer, prompt_bytes: bytes,
                           seed: int, generation_settings: dict, input_device):
    """One independent seeded inference; no prompt alteration or text repair."""
    import torch

    prompt_text = prompt_bytes.decode("utf-8")
    token_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    max_new_tokens = int(generation_settings["max_new_tokens"])
    if not token_ids or len(token_ids) > int(generation_settings["max_input_tokens"]):
        raise ValueError("Unexpected frozen input-token count")
    if len(token_ids) + max_new_tokens > int(generation_settings["max_total_tokens"]):
        raise ValueError("Frozen total-token limit exceeded")

    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    input_ids = torch.tensor([token_ids], dtype=torch.long, device=input_device)
    attention_mask = torch.ones_like(input_ids)
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        sequences = model.generate(
            input_ids=input_ids, attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=generation_settings["do_sample"],
            temperature=generation_settings["temperature"],
            top_p=generation_settings["top_p"],
            top_k=generation_settings["top_k"],
            repetition_penalty=generation_settings["repetition_penalty"],
            num_beams=generation_settings["num_beams"],
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True, prefill_chunk_size=512, cache_implementation="dynamic",
            return_dict_in_generate=False, output_scores=False,
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    generated_ids = sequences[0, len(token_ids):].detach().cpu().tolist()
    output_text = tokenizer.decode(
        generated_ids, skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    eos_ids = tokenizer.eos_token_id
    if not isinstance(eos_ids, (tuple, list, set)):
        eos_ids = [eos_ids]
    finish_reason = (
        "eos" if generated_ids and generated_ids[-1] in eos_ids
        else "max_new_tokens" if len(generated_ids) >= max_new_tokens
        else "stopped_other"
    )
    return output_text.encode("utf-8"), {
        "input_tokens": len(token_ids), "generated_tokens": len(generated_ids),
        "finish_reason": finish_reason, "duration_seconds": round(elapsed, 4),
        "generated_token_ids_sha256": module_hash(json.dumps(
            generated_ids, separators=(",", ":")
        ).encode("utf-8")),
    }


def refresh_attempt_log(task_list, checkpoint_directory, destination):
    """Atomic complete ledger built only from durable per-task journals."""
    records = []
    for task in task_list:
        path = checkpoint_directory / (task["task_id"] + ".json")
        if path.is_file():
            record = json.loads(path.read_text(encoding="utf-8"))
            records.append({
                "task_id": task["task_id"], "state": record["state"],
                "attempts": record["attempts"],
                "raw_sha256": record.get("raw_sha256"),
                "generated_tokens": record.get("generated_tokens"),
                "finish_reason": record.get("finish_reason"),
                "diagnostic": record.get("diagnostic"),
            })
    atomic_replace_bytes(destination, b"".join(module_json_bytes(x) for x in records))


# END_STAGE07_GENERATION_MODULE

from pathlib import Path
import importlib.util
import json
import time


CELL07_STARTED = time.time()
require_cell_pass(6)
shell = get_ipython()
if shell is None:
    raise RuntimeError("Cell 07 requires a Jupyter/IPython kernel")
start_marker = "# BEGIN_STAGE07_" + "GENERATION_MODULE"
end_marker = "# END_STAGE07_" + "GENERATION_MODULE"
matches = [s for s in reversed(shell.history_manager.input_hist_raw[-12:])
           if start_marker in s and end_marker in s]
if not matches or matches[0].count(start_marker) != 1 or matches[0].count(end_marker) != 1:
    raise RuntimeError("Could not uniquely recover the visible generation module source")
module_source = (matches[0].split(start_marker, 1)[1]
                 .split(end_marker, 1)[0].strip() + "\n")
generation_module_path = PROJECT_ROOT / "src/generation.py"
write_bytes_immutable(generation_module_path, module_source.encode("utf-8"))
spec = importlib.util.spec_from_file_location("stage07_generation", generation_module_path)
if spec is None or spec.loader is None:
    raise RuntimeError("Could not load frozen generation module")
generation = importlib.util.module_from_spec(spec)
spec.loader.exec_module(generation)

test_cases = [
    ("import pytest\ndef test_ok():\n    assert True\n", "accepted_raw_python"),
    ("```python\nassert True\n```", "accepted_outer_python_fence"),
    ("Text:\n```python\nassert True\n```", "rejected"),
    ("```python\nassert True\n```\n```python\nassert False\n```", "rejected"),
    ("```text\nassert True\n```", "rejected"),
    ("  \n", "rejected"),
]
for content, expected in test_cases:
    _, observed, _ = generation.extract_python_module(content)
    if observed != expected:
        raise RuntimeError("Stage 00 extraction fixture mismatch")
if read_json(PROJECT_ROOT / "config/extraction_policy.json")["policy_version"] != "1.0":
    raise RuntimeError("Stage 00 frozen extraction policy differs")
if read_json(PROJECT_ROOT / "reports/qualification/stage00_extraction.json")["status"] != "PASS":
    raise RuntimeError("Stage 00 extraction fixtures were not qualified")

TEST_TASK = TASKS[0]
TEST_FINGERPRINT = "test_generation_fingerprint"
TEST_RECORD = {
    "task_id": TEST_TASK["task_id"], "task_sha256": generation.module_hash(
        generation.module_json_bytes(TEST_TASK)
    ), "prompt_sha256": TEST_TASK["prompt_sha256"],
    "expected_output_path": TEST_TASK["expected_output_path"],
    "fingerprint": TEST_FINGERPRINT, "state": "STARTED", "attempts": [],
}
generation.validate_task_checkpoint(TEST_RECORD, TEST_TASK, TEST_FINGERPRINT)
try:
    generation.validate_task_checkpoint(TEST_RECORD, TEST_TASK, "changed_fingerprint")
except RuntimeError:
    pass
else:
    raise RuntimeError("Stale-checkpoint rejection fixture failed")

write_json_immutable(PROJECT_ROOT / "reports/qualification/generation_module.json", {
    "status": "PASS", "module_sha256": sha256_file(generation_module_path),
    "extraction_fixtures": len(test_cases), "checkpoint_binding_fixtures": 2,
})
report_cell(7, module_sha256=sha256_file(generation_module_path),
            extraction_policy="stage00-v1-frozen", fixture_count=8)


{
  "status": "PASS",
  "notebook": "07",
  "cell": 7,
  "version": "stage07-fresh-pilot-v1",
  "module_sha256": "1b70bd309c09bb863a364e3eb4a12689c909ffb23c73819770a4c1232903bcf1",
  "extraction_policy": "stage00-v1-frozen",
  "fixture_count": 8
}

NOTEBOOK 07 — CELL 07: PASS


{'status': 'PASS',
 'notebook': '07',
 'cell': 7,
 'version': 'stage07-fresh-pilot-v1',
 'module_sha256': '1b70bd309c09bb863a364e3eb4a12689c909ffb23c73819770a4c1232903bcf1',
 'extraction_policy': 'stage00-v1-frozen',
 'fixture_count': 8}

In [8]:
# ============================================================
# NOTEBOOK 07 — CELL 08
# Freeze run fingerprint and qualify hash-bound checkpoint resumption.
# ============================================================

from __future__ import annotations

from pathlib import Path
import json
import os
import time


CELL08_STARTED = time.time()
require_cell_pass(7)
GENERATION_CONFIG = {
    "version": STAGE07_VERSION,
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "tokenizer_revision": TOKENIZER_REVISION,
    "dtype": RUNTIME["dtype"],
    "max_input_tokens": int(RUNTIME["max_rendered_input_tokens"]),
    "max_new_tokens": int(RUNTIME["max_new_tokens"]),
    "max_total_tokens": int(RUNTIME["max_total_tokens"]),
    "generation": GENERATION_SETTINGS,
    "decoding": {"skip_special_tokens": True,
                 "clean_up_tokenization_spaces": False},
    "environment_packages": PACKAGE_VERSIONS,
    "device_map": MODEL_LOAD_RECORD["device_map"],
    "source_manifests": {
        stage: {key: value[key] for key in (
            "internal_manifest_sha256", "external_manifest_sha256"
        )} for stage, value in SOURCE_IDENTITIES.items()
    },
    "task_ledger_sha256": sha256_bytes(TASK_LEDGER_BYTES),
    "generation_module_sha256": sha256_file(generation_module_path),
}
GENERATION_FINGERPRINT = sha256_bytes(canonical_json_bytes(GENERATION_CONFIG))
GENERATION_CONFIG["run_fingerprint"] = GENERATION_FINGERPRINT
write_json_immutable(PROJECT_ROOT / "config/generation_frozen.json", GENERATION_CONFIG)

CHECKPOINT_DIRECTORY = PROJECT_ROOT / "checkpoints/tasks"
CHECKPOINT_DIRECTORY.mkdir(parents=True, exist_ok=True)
RAW_DIRECTORY = PROJECT_ROOT / "runs/pilot/raw"
RAW_DIRECTORY.mkdir(parents=True, exist_ok=True)
LOG_PATH = PROJECT_ROOT / "logs/generation.jsonl"

# Optional durable resume from a prior saved Kaggle notebook output dataset.
# The user may set STAGE07_RESUME_ROOT to the previous project directory.
RESUME_SOURCE = os.environ.get("STAGE07_RESUME_ROOT", "").strip()
if RESUME_SOURCE:
    previous_project = Path(RESUME_SOURCE)
    previous_config_path = previous_project / "config/generation_frozen.json"
    if not previous_config_path.is_file():
        raise RuntimeError("RESUME_BLOCK: previous Stage 07 configuration missing")
    if read_json(previous_config_path).get("run_fingerprint") != GENERATION_FINGERPRINT:
        raise RuntimeError("RESUME_BLOCK: prior checkpoint fingerprint differs")
    expected_checkpoint_files = {f"{r['task_id']}.json" for r in TASKS}
    expected_raw_files = {f"{r['task_id']}.txt" for r in TASKS}
    for relative_dir, allowed in (("checkpoints/tasks", expected_checkpoint_files),
                                  ("runs/pilot/raw", expected_raw_files)):
        previous_dir = previous_project / relative_dir
        if previous_dir.is_dir():
            for src in previous_dir.iterdir():
                if src.name not in allowed or not src.is_file() or src.is_symlink():
                    raise RuntimeError("RESUME_BLOCK: unexpected prior journal/output file")
                dest = PROJECT_ROOT / relative_dir / src.name
                write_bytes_immutable(dest, src.read_bytes())

expected_task_ids = {task["task_id"] for task in TASKS}
unexpected_checkpoints = {
    p.stem for p in CHECKPOINT_DIRECTORY.glob("*.json")
} - expected_task_ids
unexpected_outputs = {
    p.stem for p in RAW_DIRECTORY.glob("*.txt")
} - expected_task_ids
if unexpected_checkpoints or unexpected_outputs:
    raise RuntimeError("Unknown task checkpoints or raw output files present")

resumed = 0
recovered_raw = 0
interrupted_without_output = 0
for task in TASKS:
    checkpoint_path = CHECKPOINT_DIRECTORY / (task["task_id"] + ".json")
    raw_path = PROJECT_ROOT / task["expected_output_path"]
    if checkpoint_path.is_file():
        record = read_json(checkpoint_path)
        generation.validate_task_checkpoint(record, task, GENERATION_FINGERPRINT)
        if record["state"] in ("GENERATED", "RAW_RECOVERED"):
            if not raw_path.is_file() or sha256_file(raw_path) != record["raw_sha256"]:
                raise RuntimeError("Completed checkpoint raw hash is invalid")
            resumed += 1
        elif record["state"] == "NO_OUTPUT_RUNTIME_FAILURE":
            if raw_path.exists():
                raise RuntimeError("No-output checkpoint unexpectedly has raw bytes")
            resumed += 1
        elif record["state"] == "STARTED":
            if raw_path.is_file():
                record.update({"state": "RAW_RECOVERED", "raw_sha256": sha256_file(raw_path),
                               "raw_bytes": raw_path.stat().st_size,
                               "generated_tokens": None, "finish_reason": None,
                               "diagnostic": "Raw bytes survived interruption; generation metadata unavailable"})
                generation.write_task_checkpoint(checkpoint_path, record)
                recovered_raw += 1
            else:
                interrupted_without_output += 1
    elif raw_path.is_file():
        record = {
            "task_id": task["task_id"], "task_sha256": generation.module_hash(
                generation.module_json_bytes(task)),
            "fingerprint": GENERATION_FINGERPRINT,
            "prompt_sha256": task["prompt_sha256"],
            "expected_output_path": task["expected_output_path"],
            "state": "RAW_RECOVERED", "attempts": [],
            "raw_sha256": sha256_file(raw_path), "raw_bytes": raw_path.stat().st_size,
            "generated_tokens": None, "finish_reason": None,
            "diagnostic": "Recovered orphan raw bytes; metadata unavailable",
        }
        generation.write_task_checkpoint(checkpoint_path, record)
        recovered_raw += 1

generation.refresh_attempt_log(TASKS, CHECKPOINT_DIRECTORY, LOG_PATH)
write_json_atomic(PROJECT_ROOT / "reports/audit/resumption_preflight.json", {
    "status": "PASS", "run_fingerprint": GENERATION_FINGERPRINT,
    "completed_checkpoints_reused": resumed, "orphan_raw_recovered": recovered_raw,
    "interrupted_without_raw_output": interrupted_without_output,
})
report_cell(8, run_fingerprint=GENERATION_FINGERPRINT,
            completed_checkpoints_reused=resumed, orphan_raw_recovered=recovered_raw,
            interrupted_without_output=interrupted_without_output)


{
  "status": "PASS",
  "notebook": "07",
  "cell": 8,
  "version": "stage07-fresh-pilot-v1",
  "run_fingerprint": "c264c98229d83a258c4620fdb761d5d22b792915baf2d39043e8b4653c4cd0fb",
  "completed_checkpoints_reused": 0,
  "orphan_raw_recovered": 0,
  "interrupted_without_output": 0
}

NOTEBOOK 07 — CELL 08: PASS


{'status': 'PASS',
 'notebook': '07',
 'cell': 8,
 'version': 'stage07-fresh-pilot-v1',
 'run_fingerprint': 'c264c98229d83a258c4620fdb761d5d22b792915baf2d39043e8b4653c4cd0fb',
 'completed_checkpoints_reused': 0,
 'orphan_raw_recovered': 0,
 'interrupted_without_output': 0}

In [9]:
# ============================================================
# NOTEBOOK 07 — CELL 09
# Generate 200 tasks sequentially; checkpoint raw bytes after each task.
# ============================================================

from __future__ import annotations

import gc
import json
import time


CELL09_STARTED = time.time()
require_cell_pass(8)
GENERATION_ARGUMENTS = {
    **GENERATION_SETTINGS,
    "max_input_tokens": GENERATION_CONFIG["max_input_tokens"],
    "max_new_tokens": GENERATION_CONFIG["max_new_tokens"],
    "max_total_tokens": GENERATION_CONFIG["max_total_tokens"],
}


def is_permitted_no_output_retry(error):
    """Exactly one same-settings retry for a known GPU/runtime no-output failure."""
    if isinstance(error, (torch.cuda.OutOfMemoryError, OSError)):
        return True
    if isinstance(error, RuntimeError):
        message = str(error).lower()
        return ("cuda error" in message or "cublas_status_alloc_failed" in message
                or "device-side assert" in message)
    return False


new_generated = 0
new_no_output = 0
reused_outcomes = 0
for index, task in enumerate(TASKS, 1):
    checkpoint_path = CHECKPOINT_DIRECTORY / (task["task_id"] + ".json")
    raw_path = PROJECT_ROOT / task["expected_output_path"]
    if checkpoint_path.is_file():
        record = read_json(checkpoint_path)
        generation.validate_task_checkpoint(record, task, GENERATION_FINGERPRINT)
        if record["state"] in ("GENERATED", "RAW_RECOVERED", "NO_OUTPUT_RUNTIME_FAILURE"):
            reused_outcomes += 1
            continue
    else:
        record = {
            "task_id": task["task_id"],
            "task_sha256": generation.module_hash(generation.module_json_bytes(task)),
            "fingerprint": GENERATION_FINGERPRINT,
            "prompt_sha256": task["prompt_sha256"],
            "expected_output_path": task["expected_output_path"],
            "state": "STARTED", "attempts": [],
        }

    if raw_path.is_file():
        raise RuntimeError("Existing raw output must be reconciled before generation")
    if len(record["attempts"]) >= 2:
        # Both attempts were interrupted without durable output. Preserve failure.
        record.update({"state": "NO_OUTPUT_RUNTIME_FAILURE",
                       "diagnostic": "Two recorded attempts ended without raw output",
                       "generated_tokens": None, "finish_reason": None})
        generation.write_task_checkpoint(checkpoint_path, record)
        generation.refresh_attempt_log(TASKS, CHECKPOINT_DIRECTORY, LOG_PATH)
        new_no_output += 1
        continue

    prompt_bytes = (PROJECT_ROOT / task["prompt_path"]).read_bytes()
    if sha256_bytes(prompt_bytes) != task["prompt_sha256"]:
        raise RuntimeError("Frozen task prompt changed before generation")

    while len(record["attempts"]) < 2:
        attempt_number = len(record["attempts"]) + 1
        attempt = {"number": attempt_number, "status": "STARTED",
                   "seed": int(task["seed"]),
                   "generation_fingerprint": GENERATION_FINGERPRINT}
        record["attempts"].append(attempt)
        record["state"] = "STARTED"
        generation.write_task_checkpoint(checkpoint_path, record)
        generation.refresh_attempt_log(TASKS, CHECKPOINT_DIRECTORY, LOG_PATH)
        output_was_produced = False
        try:
            raw_bytes, generation_metadata = generation.generated_continuation(
                model=model, tokenizer=tokenizer, prompt_bytes=prompt_bytes,
                seed=int(task["seed"]), generation_settings=GENERATION_ARGUMENTS,
                input_device=embedding_device,
            )
            output_was_produced = True
            # Raw bytes are immutable, and committed before any success metadata.
            generation.immutable_bytes(raw_path, raw_bytes)
            attempt.update({"status": "GENERATED", **generation_metadata})
            record.update({
                "state": "GENERATED", "raw_sha256": sha256_bytes(raw_bytes),
                "raw_bytes": len(raw_bytes), "generated_tokens": generation_metadata["generated_tokens"],
                "input_tokens": generation_metadata["input_tokens"],
                "finish_reason": generation_metadata["finish_reason"],
                "duration_seconds": generation_metadata["duration_seconds"],
                "generated_token_ids_sha256": generation_metadata["generated_token_ids_sha256"],
                "diagnostic": None,
            })
            generation.write_task_checkpoint(checkpoint_path, record)
            new_generated += 1
            break
        except Exception as error:
            # A source, code, or configuration defect must not become a model score.
            if output_was_produced or raw_path.is_file():
                raise RuntimeError(
                    "OUTPUT_COMMIT_BLOCK: generated bytes existed, but saving/metadata "
                    "failed. Do not regenerate this task; recover the original bytes."
                ) from error
            if not is_permitted_no_output_retry(error):
                raise RuntimeError(
                    "SYSTEM_BLOCK: unexpected generation implementation error at "
                    + task["task_id"] + ": " + type(error).__name__ + " " + str(error)[:500]
                ) from error
            attempt.update({"status": "NO_OUTPUT_RUNTIME_FAILURE",
                            "error_type": type(error).__name__,
                            "diagnostic": str(error)[:500]})
            record["diagnostic"] = type(error).__name__ + ": " + str(error)[:500]
            if len(record["attempts"]) >= 2:
                record.update({"state": "NO_OUTPUT_RUNTIME_FAILURE",
                               "generated_tokens": None, "finish_reason": None})
                generation.write_task_checkpoint(checkpoint_path, record)
                new_no_output += 1
                break
            generation.write_task_checkpoint(checkpoint_path, record)
            torch.cuda.empty_cache()
            gc.collect()
        finally:
            generation.refresh_attempt_log(TASKS, CHECKPOINT_DIRECTORY, LOG_PATH)

    if index % 10 == 0 or index == 1 or index == 200:
        print({"completed_position": index, "total": 200,
               "new_generated": new_generated, "new_no_output": new_no_output,
               "resumed": reused_outcomes})

report_cell(9, final_task_count=200, newly_generated=new_generated,
            new_no_output_runtime_failures=new_no_output,
            reused_existing_outcomes=reused_outcomes,
            all_tasks_checkpointed=True)


{'completed_position': 1, 'total': 200, 'new_generated': 1, 'new_no_output': 0, 'resumed': 0}
{'completed_position': 10, 'total': 200, 'new_generated': 10, 'new_no_output': 0, 'resumed': 0}
{'completed_position': 20, 'total': 200, 'new_generated': 20, 'new_no_output': 0, 'resumed': 0}
{'completed_position': 30, 'total': 200, 'new_generated': 30, 'new_no_output': 0, 'resumed': 0}
{'completed_position': 40, 'total': 200, 'new_generated': 40, 'new_no_output': 0, 'resumed': 0}
{'completed_position': 50, 'total': 200, 'new_generated': 50, 'new_no_output': 0, 'resumed': 0}
{'completed_position': 60, 'total': 200, 'new_generated': 60, 'new_no_output': 0, 'resumed': 0}
{'completed_position': 70, 'total': 200, 'new_generated': 70, 'new_no_output': 0, 'resumed': 0}
{'completed_position': 80, 'total': 200, 'new_generated': 80, 'new_no_output': 0, 'resumed': 0}
{'completed_position': 90, 'total': 200, 'new_generated': 90, 'new_no_output': 0, 'resumed': 0}
{'completed_position': 100, 'total': 200, 

{'status': 'PASS',
 'notebook': '07',
 'cell': 9,
 'version': 'stage07-fresh-pilot-v1',
 'final_task_count': 200,
 'newly_generated': 200,
 'new_no_output_runtime_failures': 0,
 'reused_existing_outcomes': 0,
 'all_tasks_checkpointed': True}

In [10]:
# ============================================================
# NOTEBOOK 07 — CELL 10
# Reconcile one durable final outcome for each of 200 tasks.
# ============================================================

from __future__ import annotations

from collections import Counter
import json
import time


CELL10_STARTED = time.time()
require_cell_pass(9)
OUTCOME_RECORDS = []
for task in TASKS:
    checkpoint_path = CHECKPOINT_DIRECTORY / (task["task_id"] + ".json")
    if not checkpoint_path.is_file():
        raise RuntimeError("Missing task outcome checkpoint: " + task["task_id"])
    record = read_json(checkpoint_path)
    generation.validate_task_checkpoint(record, task, GENERATION_FINGERPRINT)
    if record["state"] == "STARTED":
        raise RuntimeError("Task still interrupted: " + task["task_id"])
    raw_path = PROJECT_ROOT / task["expected_output_path"]
    if record["state"] in ("GENERATED", "RAW_RECOVERED"):
        if (not raw_path.is_file()
                or sha256_file(raw_path) != record["raw_sha256"]
                or raw_path.stat().st_size != record["raw_bytes"]):
            raise RuntimeError("Raw bytes do not match the final task checkpoint")
    elif raw_path.exists():
        raise RuntimeError("Unexpected raw bytes for a no-output task")
    OUTCOME_RECORDS.append({"task_id": task["task_id"],
                            "state": record["state"], "attempt_count": len(record["attempts"]),
                            "raw_sha256": record.get("raw_sha256"),
                            "generated_tokens": record.get("generated_tokens"),
                            "finish_reason": record.get("finish_reason")})
if len(OUTCOME_RECORDS) != 200 or len({r["task_id"] for r in OUTCOME_RECORDS}) != 200:
    raise RuntimeError("Final generation population is not exactly 200 tasks")
OUTCOME_COUNTS = dict(Counter(r["state"] for r in OUTCOME_RECORDS))
write_json_immutable(PROJECT_ROOT / "reports/audit/task_outcome_reconciliation.json", {
    "status": "PASS", "task_count": 200, "outcome_counts": OUTCOME_COUNTS,
    "raw_present_count": sum(r["raw_sha256"] is not None for r in OUTCOME_RECORDS),
    "no_quality_retry": True, "model_outputs_repaired": False,
})
report_cell(10, task_count=200, outcomes=OUTCOME_COUNTS,
            raw_outputs_present=sum(r["raw_sha256"] is not None for r in OUTCOME_RECORDS))


{
  "status": "PASS",
  "notebook": "07",
  "cell": 10,
  "version": "stage07-fresh-pilot-v1",
  "task_count": 200,
  "outcomes": {
    "GENERATED": 200
  },
  "raw_outputs_present": 200
}

NOTEBOOK 07 — CELL 10: PASS


{'status': 'PASS',
 'notebook': '07',
 'cell': 10,
 'version': 'stage07-fresh-pilot-v1',
 'task_count': 200,
 'outcomes': {'GENERATED': 200},
 'raw_outputs_present': 200}

In [11]:
# ============================================================
# NOTEBOOK 07 — CELL 11
# Extract Python under frozen Stage 00 policy; never modify raw output.
# ============================================================

from __future__ import annotations

from collections import Counter
import hashlib
import json
import time


CELL11_STARTED = time.time()
require_cell_pass(10)
EXTRACTION_RECORDS = []
for index, task in enumerate(TASKS, 1):
    checkpoint = read_json(CHECKPOINT_DIRECTORY / (task["task_id"] + ".json"))
    raw_sha = checkpoint.get("raw_sha256")
    result = {"task_id": task["task_id"], "raw_sha256": raw_sha,
              "artifact_id": None, "extraction_status": "no_output",
              "extracted_sha256": None, "diagnostic": checkpoint.get("diagnostic")}
    if raw_sha is not None:
        raw_path = PROJECT_ROOT / task["expected_output_path"]
        raw_bytes = raw_path.read_bytes()
        if sha256_bytes(raw_bytes) != raw_sha:
            raise RuntimeError("Raw bytes changed during extraction")
        raw_text = raw_bytes.decode("utf-8", errors="strict")
        code, extraction_status, diagnostic = generation.extract_python_module(raw_text)
        artifact_id = "art_" + sha256_bytes(
            (task["task_id"] + "|" + raw_sha).encode("utf-8")
        )[:24]
        result.update({"artifact_id": artifact_id,
                       "extraction_status": extraction_status, "diagnostic": diagnostic})
        if code is not None:
            code_bytes = code.encode("utf-8")
            extracted_path = PROJECT_ROOT / "extracted" / artifact_id / "test_generated.py"
            write_bytes_immutable(extracted_path, code_bytes)
            result["extracted_sha256"] = sha256_bytes(code_bytes)
        # Extraction never alters or removes the original generation output.
        if sha256_file(raw_path) != raw_sha:
            raise RuntimeError("Extraction unexpectedly changed raw bytes")
    EXTRACTION_RECORDS.append(result)
    if index % 40 == 0:
        print({"extraction_checked": index, "expected": 200})

write_bytes_immutable(
    PROJECT_ROOT / "data/audits/extraction_ledger.jsonl",
    b"".join(canonical_json_bytes(r) for r in EXTRACTION_RECORDS),
)
EXTRACTION_COUNTS = dict(Counter(r["extraction_status"] for r in EXTRACTION_RECORDS))
report_cell(11, task_count=200, extraction_status_counts=EXTRACTION_COUNTS,
            raw_outputs_preserved=True, code_repairs_performed=False)


{'extraction_checked': 40, 'expected': 200}
{'extraction_checked': 80, 'expected': 200}
{'extraction_checked': 120, 'expected': 200}
{'extraction_checked': 160, 'expected': 200}
{'extraction_checked': 200, 'expected': 200}
{
  "status": "PASS",
  "notebook": "07",
  "cell": 11,
  "version": "stage07-fresh-pilot-v1",
  "task_count": 200,
  "extraction_status_counts": {
    "rejected": 94,
    "accepted_outer_python_fence": 106
  },
  "raw_outputs_preserved": true,
  "code_repairs_performed": false
}

NOTEBOOK 07 — CELL 11: PASS


{'status': 'PASS',
 'notebook': '07',
 'cell': 11,
 'version': 'stage07-fresh-pilot-v1',
 'task_count': 200,
 'extraction_status_counts': {'rejected': 94,
  'accepted_outer_python_fence': 106},
 'raw_outputs_preserved': True,
 'code_repairs_performed': False}

In [12]:
# ============================================================
# NOTEBOOK 07 — CELL 12
# Freeze artifacts.csv, generation statistics and internal manifest.
# ============================================================

from __future__ import annotations

from collections import Counter
import csv
import io
import json
import statistics
import time


CELL12_STARTED = time.time()
require_cell_pass(11)
EXTRACTION_LOOKUP = {row["task_id"]: row for row in EXTRACTION_RECORDS}
ARTIFACT_ROWS = []
for task in TASKS:
    outcome = read_json(CHECKPOINT_DIRECTORY / (task["task_id"] + ".json"))
    extraction = EXTRACTION_LOOKUP[task["task_id"]]
    ARTIFACT_ROWS.append({
        "task_id": task["task_id"],
        "artifact_id": extraction["artifact_id"] or "",
        "raw_hash": extraction["raw_sha256"] or "",
        "extraction_status": extraction["extraction_status"],
        "extracted_hash": extraction["extracted_sha256"] or "",
        "generated_tokens": ("" if outcome.get("generated_tokens") is None
                             else outcome["generated_tokens"]),
        "finish_reason": outcome.get("finish_reason") or "",
        "attempts": len(outcome["attempts"]),
        "diagnostic": extraction.get("diagnostic") or outcome.get("diagnostic") or "",
    })
ARTIFACT_COLUMNS = (
    "task_id", "artifact_id", "raw_hash", "extraction_status", "extracted_hash",
    "generated_tokens", "finish_reason", "attempts", "diagnostic",
)
buffer = io.StringIO(newline="")
writer = csv.DictWriter(buffer, fieldnames=ARTIFACT_COLUMNS, lineterminator="\n")
writer.writeheader()
writer.writerows(ARTIFACT_ROWS)
ARTIFACTS_CSV_PATH = PROJECT_ROOT / "data/artifacts.csv"
write_bytes_immutable(ARTIFACTS_CSV_PATH, buffer.getvalue().encode("utf-8"))
if len(ARTIFACT_ROWS) != 200 or len({r["task_id"] for r in ARTIFACT_ROWS}) != 200:
    raise RuntimeError("artifacts.csv does not reconcile 200 unique tasks")

duration_values = []
token_values = []
attempt_counts = Counter()
finish_counts = Counter()
for task in TASKS:
    record = read_json(CHECKPOINT_DIRECTORY / (task["task_id"] + ".json"))
    attempt_counts[len(record["attempts"])] += 1
    finish_counts[record.get("finish_reason") or "unavailable"] += 1
    if record.get("generated_tokens") is not None:
        token_values.append(int(record["generated_tokens"]))
    if record.get("duration_seconds") is not None:
        duration_values.append(float(record["duration_seconds"]))


def describe_values(values):
    if not values:
        return {"count": 0, "mean": None, "median": None, "minimum": None, "maximum": None}
    return {"count": len(values), "mean": round(statistics.mean(values), 3),
            "median": round(statistics.median(values), 3),
            "minimum": min(values), "maximum": max(values)}


GENERATION_SUMMARY = {
    "status": "PASS", "task_count": 200,
    "raw_output_count": sum(bool(r["raw_hash"]) for r in ARTIFACT_ROWS),
    "extracted_module_count": sum(bool(r["extracted_hash"]) for r in ARTIFACT_ROWS),
    "extraction_status_counts": dict(Counter(r["extraction_status"] for r in ARTIFACT_ROWS)),
    "attempt_count_distribution": dict(sorted(attempt_counts.items())),
    "finish_reason_counts": dict(finish_counts),
    "generated_token_distribution": describe_values(token_values),
    "generation_duration_seconds": describe_values(duration_values),
    "model_revision": MODEL_REVISION,
    "generation_fingerprint": GENERATION_FINGERPRINT,
    "raw_model_outputs_repaired": False, "quality_based_retries": False,
}
if GENERATION_SUMMARY["raw_output_count"] == 0:
    raise RuntimeError("GENERATION_BLOCK: no model output was preserved")
write_json_immutable(PROJECT_ROOT / "reports/audit/generation_summary.json", GENERATION_SUMMARY)

PACKAGE_RELATIVE_PATHS = [
    "config/generation_frozen.json", "config/extraction_policy.json",
    "config/analysis_registered.yaml", "config/metrics.yaml",
    "src/generation.py", "data/artifacts.csv", "data/audits/extraction_ledger.jsonl",
    "logs/generation.jsonl", "registries/06_frozen_dependency_hashes.json",
    "reports/audit/upstream_verification.json",
    "reports/audit/restored_inputs.json",
    "reports/audit/resumption_preflight.json",
    "reports/audit/task_outcome_reconciliation.json",
    "reports/audit/generation_summary.json",
    "reports/runtime/environment_preflight.json",
    "reports/runtime/model_load_qualification.json",
    "reports/qualification/seeded_generation.json",
    "reports/qualification/generation_module.json",
    "reports/qualification/stage00_extraction.json",
]
PACKAGE_RELATIVE_PATHS.extend(
    f"reports/audit/cell{number:02d}_stage07.json" for number in range(1, 12)
)
PACKAGE_RELATIVE_PATHS.extend(
    f"checkpoints/tasks/{task['task_id']}.json" for task in TASKS
)
PACKAGE_RELATIVE_PATHS.extend(
    task["expected_output_path"] for task in TASKS
    if (PROJECT_ROOT / task["expected_output_path"]).is_file()
)
PACKAGE_RELATIVE_PATHS.extend(
    f"extracted/{row['artifact_id']}/test_generated.py" for row in ARTIFACT_ROWS
    if row["extracted_hash"]
)
PACKAGE_RELATIVE_PATHS = sorted(set(PACKAGE_RELATIVE_PATHS))
PACKAGE_ARTIFACTS = []
for relative in PACKAGE_RELATIVE_PATHS:
    safe_relative(relative)
    path = PROJECT_ROOT / relative
    if not path.is_file() or path.is_symlink():
        raise RuntimeError("Missing/unsafe Stage 07 package member: " + relative)
    PACKAGE_ARTIFACTS.append({"relative_path": relative, "bytes": path.stat().st_size,
                              "sha256": sha256_file(path)})

INTERNAL_MANIFEST = {
    "stage": "07", "status": "PASS", "partition": "pilot", "design_id": "openapi-fresh-pilot-v1",
    "manifest_version": "stage07-fresh-pilot-v1",
    "packaging_mode": "LEAN_NONCUMULATIVE_STAGE_PACKAGE",
    "operation_count": 100, "task_count": 200,
    "raw_output_count": GENERATION_SUMMARY["raw_output_count"],
    "extracted_module_count": GENERATION_SUMMARY["extracted_module_count"],
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "generation_fingerprint": GENERATION_FINGERPRINT,
    "task_ledger_sha256": sha256_bytes(TASK_LEDGER_BYTES),
    "upstream_manifest_sha256": {stage: sha256_file(source["registry_path"])
                                  for stage, source in STAGES.items()},
    "artifact_count": len(PACKAGE_ARTIFACTS),
    "artifacts": PACKAGE_ARTIFACTS,
    "generated_source_repaired": False,
}
register_artifact_provenance(INTERNAL_MANIFEST)
write_json_immutable(PROJECT_ROOT / "registries/07_manifest.json", INTERNAL_MANIFEST)
report_cell(12, task_count=200, raw_outputs=GENERATION_SUMMARY["raw_output_count"],
            extracted_modules=GENERATION_SUMMARY["extracted_module_count"],
            registered_artifact_count=len(PACKAGE_ARTIFACTS),
            internal_manifest_sha256=sha256_file(PROJECT_ROOT / "registries/07_manifest.json"))


{
  "status": "PASS",
  "notebook": "07",
  "cell": 12,
  "version": "stage07-fresh-pilot-v1",
  "task_count": 200,
  "raw_outputs": 200,
  "extracted_modules": 106,
  "registered_artifact_count": 536,
  "internal_manifest_sha256": "14ee76a7ce4609b40c1696c2e7cd2a7af7aaaee8a4bf2c0b7759be73967b72d3"
}

NOTEBOOK 07 — CELL 12: PASS


{'status': 'PASS',
 'notebook': '07',
 'cell': 12,
 'version': 'stage07-fresh-pilot-v1',
 'task_count': 200,
 'raw_outputs': 200,
 'extracted_modules': 106,
 'registered_artifact_count': 536,
 'internal_manifest_sha256': '14ee76a7ce4609b40c1696c2e7cd2a7af7aaaee8a4bf2c0b7759be73967b72d3'}

In [13]:
# ============================================================
# NOTEBOOK 07 — CELL 13
# Package the lean Stage 07 dataset with manifests and SHA-256 evidence.
# ============================================================

from __future__ import annotations

import json
import os
import time
import zipfile


CELL13_STARTED = time.time()
require_cell_pass(12)
ARCHIVE_NAME = "stage_07_fresh.zip"
ARCHIVE_PATH = DATASET_ROOT / ARCHIVE_NAME
ARCHIVE_PREFIX = "stage_07_fresh/prose_operation_experiment/"
manifest_relative = "registries/07_manifest.json"
all_members = PACKAGE_RELATIVE_PATHS + [manifest_relative]
archive_temp = ARCHIVE_PATH.with_name(ARCHIVE_NAME + ".tmp")
if not ARCHIVE_PATH.exists():
    with zipfile.ZipFile(archive_temp, "w", compression=zipfile.ZIP_DEFLATED,
                         compresslevel=7, allowZip64=True) as archive:
        for relative in all_members:
            entry = zipfile.ZipInfo(
                ARCHIVE_PREFIX + relative, date_time=(2026, 11, 11, 0, 0, 0)
            )
            entry.compress_type = zipfile.ZIP_DEFLATED
            archive.writestr(entry, (PROJECT_ROOT / relative).read_bytes(),
                             compress_type=zipfile.ZIP_DEFLATED, compresslevel=7)
    os.replace(archive_temp, ARCHIVE_PATH)
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    expected_names = {ARCHIVE_PREFIX + relative for relative in all_members}
    if len(archive.namelist()) != len(all_members) or set(archive.namelist()) != expected_names:
        raise RuntimeError("Stage 07 ZIP member list differs from frozen manifest")
    if archive.testzip() is not None:
        raise RuntimeError("Stage 07 ZIP CRC check failed")
    for artifact in PACKAGE_ARTIFACTS:
        if (sha256_bytes(archive.read(ARCHIVE_PREFIX + artifact["relative_path"]))
                != artifact["sha256"]):
            raise RuntimeError("Archived artifact SHA-256 mismatch: " + artifact["relative_path"])
    internal_bytes = archive.read(ARCHIVE_PREFIX + manifest_relative)
    if sha256_bytes(internal_bytes) != sha256_file(PROJECT_ROOT / manifest_relative):
        raise RuntimeError("Archived internal manifest differs")

EXTERNAL_MANIFEST = {
    "stage": "07", "status": "PASS", "partition": "pilot", "design_id": "openapi-fresh-pilot-v1",
    "archive_name": ARCHIVE_NAME, "archive_sha256": sha256_file(ARCHIVE_PATH),
    "archive_bytes": ARCHIVE_PATH.stat().st_size,
    "archive": {"filename": ARCHIVE_NAME, "sha256": sha256_file(ARCHIVE_PATH),
                "bytes": ARCHIVE_PATH.stat().st_size, "member_count": len(all_members),
                "project_prefix": ARCHIVE_PREFIX.rstrip("/")},
    "internal_manifest_sha256": sha256_bytes(internal_bytes),
    "internal_manifest": {"relative_path": manifest_relative,
                          "sha256": sha256_bytes(internal_bytes),
                          "artifact_count": len(PACKAGE_ARTIFACTS)},
    "task_count": 200, "operation_count": 100,
    "raw_output_count": GENERATION_SUMMARY["raw_output_count"],
    "extracted_module_count": GENERATION_SUMMARY["extracted_module_count"],
    "model_revision": MODEL_REVISION,
    "generation_fingerprint": GENERATION_FINGERPRINT,
    "model_outputs_repaired": False, "quality_retries": False,
    "packaging_mode": "LEAN_NONCUMULATIVE_STAGE_PACKAGE",
}
external_path = DATASET_ROOT / "stage_07_manifest.json"
write_json_immutable(external_path, EXTERNAL_MANIFEST)

# Both entries name files that actually exist at the Kaggle dataset top level.
checksum_text = (
    sha256_file(ARCHIVE_PATH) + "  " + ARCHIVE_NAME + "\n"
    + sha256_file(external_path) + "  stage_07_manifest.json\n"
)
write_bytes_immutable(DATASET_ROOT / "stage_07_checksums.sha256",
                      checksum_text.encode("utf-8"))
readme = """# Stage 07 — Pilot Raw Generations

Status: PASS

This lean dataset contains the 200-task outcome ledger, all preserved raw
model outputs, any modules accepted by the frozen Stage 00 extraction policy,
attempt logs, generation checkpoints, and frozen runtime/qualification evidence.

The expected task population remains 200 regardless of extraction success.
Raw generated bytes were not repaired, filtered, or quality-retried.

For Notebook 08, attach Stage 05, Stage 06 and Stage 07. Frozen oracle
packages are not included in this Stage 07 generation dataset.
"""
write_bytes_immutable(DATASET_ROOT / "README.md", readme.encode("utf-8"))
stage_summary = (
    "# Notebook 07 — PASS\n\n"
    "- Expected tasks: 200\n"
    f"- Preserved raw outputs: {GENERATION_SUMMARY['raw_output_count']}\n"
    f"- Extracted modules: {GENERATION_SUMMARY['extracted_module_count']}\n"
    f"- Frozen model revision: {MODEL_REVISION}\n"
    "- No quality retry or source repair occurred.\n"
    "- Ready for Notebook 08 offline execution and blind review.\n"
)
write_bytes_immutable(DATASET_ROOT / "stage_summary_for_executed.md",
                      stage_summary.encode("utf-8"))

final_report = {
    "status": "PASS", "stage": "07", "task_count": 200,
    "raw_output_count": GENERATION_SUMMARY["raw_output_count"],
    "extracted_module_count": GENERATION_SUMMARY["extracted_module_count"],
    "archive_path": str(ARCHIVE_PATH),
    "archive_sha256": EXTERNAL_MANIFEST["archive_sha256"],
    "external_manifest_sha256": sha256_file(external_path),
    "internal_manifest_sha256": EXTERNAL_MANIFEST["internal_manifest_sha256"],
    "artifact_count": len(PACKAGE_ARTIFACTS),
    "may_begin_notebook_08": True,
}
print(json.dumps(final_report, indent=2))
print("\nNOTEBOOK 07 — CELL 13: PASS")
print("STAGE 07: FROZEN — 200 task outcomes fully accounted for.")
print("NEXT: Publish Stage 07 dataset, then begin Notebook 08.")


{
  "status": "PASS",
  "stage": "07",
  "task_count": 200,
  "raw_output_count": 200,
  "extracted_module_count": 106,
  "archive_path": "/kaggle/working/stage_07_fresh_dataset/stage_07_fresh.zip",
  "archive_sha256": "ba632b9a7ad66c94efe0255a334eb57896fed98044212ee56955902ed253ae1d",
  "external_manifest_sha256": "3da3c1a7cca4cdd344d6cbaa5f1564632dc453cb937fcd8e38ff935c1836b1bb",
  "internal_manifest_sha256": "14ee76a7ce4609b40c1696c2e7cd2a7af7aaaee8a4bf2c0b7759be73967b72d3",
  "artifact_count": 536,
  "may_begin_notebook_08": true
}

NOTEBOOK 07 — CELL 13: PASS
STAGE 07: FROZEN — 200 task outcomes fully accounted for.
NEXT: Publish Stage 07 dataset, then begin Notebook 08.
